# AMPS II · Week 04 — Lab
## J2 plasticity: the return mapping at a material point, the thick tube, Newton with two tangents, the limit load

| | |
|---|---|
| **Lecture sections** | Part A (J2, return mapping, consistent tangent, tube: first yield and limit load) · Part B (quadrature spaces, Newton loop, displacement control) · Part C (verification) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | test a return mapping without any finite elements · check the tube against Lamé and the first-yield pressure · watch Newton converge quadratically with the consistent tangent and crawl with the elastic one · run into the limit load with pressure control and pass it with displacement control |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The two model modules are written once
with `%%writefile` (§1). `j2.py` is NumPy only, so ordinary cells can import it too. Results pass through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model modules  *(lecture §A2–A5, §B1–B6)*
`j2.py`: the J2 material with linear isotropic hardening in Mandel form — `update` (return mapping + tangent, for many points at once) and `uniaxial`
(one point in uniaxial stress). `plas.py`: the curved quarter-annulus mesh (as Week 3) and the 1D radial mesh, the class `Tube` with quadrature
spaces and an explicit Newton loop (`step` returns *converged, iterations, residuals*), pressure or displacement control, and the exact formulas.
Read them once; run both cells — they only write files.

In [ ]:
%%writefile /tmp/fnx/j2.py
"""J2 (von Mises) plasticity with linear isotropic hardening at the material points -- NumPy only.
Mandel vectors (xx, yy, zz, sqrt(2) xy) for stress and strain, so that sigma . eps = sigma : eps and C is a 4 x 4 matrix.
update() is the backward-Euler return mapping for MANY points at once (arrays of shape (n, 4))."""
import numpy as np

ONE = np.array([1.0, 1.0, 1.0, 0.0])                 # the identity tensor in Mandel form
IDEV = np.eye(4) - np.outer(ONE, ONE) / 3.0          # deviatoric projector: s = IDEV @ sigma

def dev(sig):
    return sig @ IDEV                                # IDEV is symmetric

def von_mises(sig):
    s = dev(sig)
    return np.sqrt(1.5 * np.einsum("ij,ij->i", s, s))

class J2:
    """E, nu: elasticity; sig0: initial yield stress; H: linear isotropic hardening modulus, sigma_y = sig0 + H p"""
    def __init__(self, E, nu, sig0, H=0.0):
        self.E, self.nu, self.sig0, self.H = E, nu, sig0, H
        self.mu = E / (2 * (1 + nu)); self.lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
        self.C = self.lmbda * np.outer(ONE, ONE) + 2 * self.mu * np.eye(4)

    def update(self, deps, sig_n, p_n, tangent="consistent"):
        """deps: strain increment since the last CONVERGED step, sig_n, p_n: stress and p at that step.
        Returns sig, p, Ct (n, 4, 4) and the mask of plastic points. tangent = "consistent" or "elastic"."""
        mu, H = self.mu, self.H
        sig_tr = sig_n + deps @ self.C                   # elastic trial stress
        s = dev(sig_tr)
        seq = np.sqrt(1.5 * np.einsum("ij,ij->i", s, s))
        f = seq - (self.sig0 + H * p_n)                  # trial yield function
        pl = f > 0.0
        dp = np.where(pl, f / (3 * mu + H), 0.0)          # plastic multiplier = increment of p
        safe = np.where(seq > 0.0, seq, 1.0)
        nhat = s / (np.sqrt(2.0 / 3.0) * safe)[:, None]  # unit normal s/|s|
        sig = sig_tr - (2 * mu * np.sqrt(1.5) * dp)[:, None] * nhat   # radial return
        Ct = np.broadcast_to(self.C, (len(f), 4, 4)).copy()
        if tangent == "consistent":
            b = 3 * mu * dp / safe                       # 1 - theta
            tb = np.where(pl, 3 * mu / (3 * mu + H) - b, 0.0)
            Ct -= 2 * mu * (b[:, None, None] * IDEV + tb[:, None, None] * np.einsum("ni,nj->nij", nhat, nhat))
        return sig, p_n + dp, Ct, pl

def uniaxial(mat, exx, tangent="consistent", tol=1e-10, maxit=50):
    """uniaxial STRESS test of one material point: prescribed history of eps_xx, sigma_yy = sigma_zz = sigma_xy = 0 by a local
    Newton iteration on the three other strain components (with the tangent of update()). Returns columns sigma_xx, p, iterations."""
    sig, p, eps, out = np.zeros((1, 4)), np.zeros(1), np.zeros(4), []
    for e in exx:
        deps = np.zeros(4); deps[0] = e - eps[0]; deps[1] = deps[2] = -mat.nu * deps[0]     # elastic guess
        for it in range(maxit):
            s, pn, Ct, pl = mat.update(deps[None], sig, p, tangent)
            r = s[0, 1:]
            if np.abs(r).max() <= tol * mat.sig0:
                break
            deps[1:] -= np.linalg.solve(Ct[0][1:, 1:], r)
        eps, sig, p = eps + deps, s, pn
        out.append((s[0, 0], pn[0], it))
    return np.array(out)

In [ ]:
%%writefile /tmp/fnx/plas.py
"""Small-strain J2 plasticity in DOLFINx 0.11: quadrature spaces for stress, p and the tangent, an explicit Newton loop.
Two geometries of the same thick tube in plane strain: the 2D quarter cross-section (curved mapped mesh, no Gmsh) and
the 1D radial model (an interval in r, weight r) used as the reference. Load: inner pressure or inner radial displacement."""
import os, sys
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_matrix, create_vector, set_bc
sys.path.insert(0, "/tmp/fnx")
from j2 import J2, ONE, IDEV, dev, von_mises

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, LEFT = 1, 2, 3, 4

def annulus_mesh(ri, ro, nr, nt):
    """quarter annulus, nr x nt cells in (r, theta) split in triangles, 6-node curved cells mapped from (r, theta) (as Week 3).
    Facet tags: 1 r = ri, 2 r = ro, 3 y = 0, 4 x = 0."""
    if comm.rank == 0:
        nI, nJ = 2 * nr + 1, 2 * nt + 1
        R, TH = np.meshgrid(np.linspace(ri, ro, nI), np.linspace(0, np.pi / 2, nJ), indexing="ij")
        x = np.c_[(R * np.cos(TH)).ravel(), (R * np.sin(TH)).ravel()]
        x[np.abs(x) < 1e-14] = 0.0
        node = lambda p: p[0] * nJ + p[1]
        mid = lambda p, q: ((p[0] + q[0]) // 2, (p[1] + q[1]) // 2)
        cells = []
        for a in range(nr):
            for b in range(nt):
                i, j = 2 * a, 2 * b
                for p, q, s in (((i, j), (i + 2, j), (i + 2, j + 2)), ((i, j), (i + 2, j + 2), (i, j + 2))):
                    cells.append([node(p), node(q), node(s), node(mid(q, s)), node(mid(p, s)), node(mid(p, q))])
        cells = np.array(cells, dtype=np.int64)
    else:
        x, cells = np.zeros((0, 2)), np.zeros((0, 6), dtype=np.int64)
    msh = mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))), x)
    rad = lambda X: np.hypot(X[0], X[1])
    return msh, _tags(msh, [(INNER, lambda X: np.isclose(rad(X), ri)), (OUTER, lambda X: np.isclose(rad(X), ro)),
                            (BOTTOM, lambda X: np.isclose(X[1], 0.0)), (LEFT, lambda X: np.isclose(X[0], 0.0))])

def radial_mesh(ri, ro, n):
    """interval ri <= r <= ro (the 1D radial model). Facet (= point) tags: 1 r = ri, 2 r = ro."""
    msh = mesh.create_interval(comm, n, [ri, ro])
    return msh, _tags(msh, [(INNER, lambda X: np.isclose(X[0], ri)), (OUTER, lambda X: np.isclose(X[0], ro))])

def _tags(msh, marks):
    fdim = msh.topology.dim - 1
    fac, val = [], []
    for tag, f in marks:
        e = mesh.locate_entities_boundary(msh, fdim, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return mesh.meshtags(msh, fdim, fac, np.concatenate(val)[idx])

# ── exact results for the tube (plane strain) ──────────────────────────────────────────────────────────────────
def lame_ur(p, ri, ro, E, nu, r=None):
    """elastic plane-strain radial displacement u_r(r) of a tube under inner pressure p (r = ri by default)"""
    r = ri if r is None else r
    A = p * ri**2 / (ro**2 - ri**2); B = A * ro**2
    return (1 + nu) / E * ((1 - 2 * nu) * A * r + B / r)

def p_first_yield(sig0, ri, ro, nu):
    """first yield at r = ri, von Mises, plane strain (sigma_zz = 2 nu A). nu = 0.5 gives sig0/sqrt(3) (1 - ri^2/ro^2)."""
    return sig0 * (ro**2 - ri**2) / np.sqrt(3 * ro**4 + (1 - 2 * nu)**2 * ri**4)

def p_limit(sig0, ri, ro):
    return 2 * sig0 / np.sqrt(3) * np.log(ro / ri)

# ── the model ──────────────────────────────────────────────────────────────────────────────────────────────────
class Tube:
    """msh, ft: annulus_mesh (2D) or radial_mesh (1D); mat: J2. qdeg: quadrature degree of the internal variables.
    control = "pressure" (inner pressure p) or "displacement" (inner radial displacement)."""
    def __init__(self, msh, ft, mat, ri, ro, deg=2, qdeg=2, control="pressure", tangent="consistent"):
        self.msh, self.mat, self.ri, self.ro, self.tangent, self.control = msh, mat, ri, ro, tangent, control
        self.dim = msh.topology.dim
        cell = msh.basix_cell()
        if self.dim == 2:
            self.V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
        else:
            self.V = fem.functionspace(msh, ("Lagrange", deg))
        Q4 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4,), degree=qdeg))
        Q44 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4, 4), degree=qdeg))
        Q1 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(), degree=qdeg))
        self.sig, self.sig_n, self.Ct = fem.Function(Q4), fem.Function(Q4), fem.Function(Q44)
        self.p, self.p_n = fem.Function(Q1), fem.Function(Q1)
        self.Ct.x.array[:] = np.tile(mat.C.ravel(), len(self.Ct.x.array) // 16)
        self.u, self.Du, self.du = fem.Function(self.V), fem.Function(self.V), fem.Function(self.V)
        self.dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": qdeg})
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
        x = ufl.SpatialCoordinate(msh)
        self.w = x[0] if self.dim == 1 else 1.0          # 1D: integrals carry r (per radian); 2D: per unit length
        pts = Q4.element.interpolation_points
        self.deps_expr = fem.Expression(self.eps4(self.Du), pts)
        self.deps = fem.Function(Q4)
        self.rq = fem.Function(Q1)                       # radius of each quadrature point
        self.rq.interpolate(fem.Expression(x[0] if self.dim == 1 else ufl.sqrt(x[0]**2 + x[1]**2), pts))
        v, du = ufl.TestFunction(self.V), ufl.TrialFunction(self.V)
        self.pload = fem.Constant(msh, 0.0)
        n = ufl.FacetNormal(msh)
        if self.dim == 2:
            Fext = -self.pload * ufl.dot(n, v) * self.ds(INNER)       # pressure: traction -p n on r = ri
        else:
            Fext = self.pload * self.ri * v * self.ds(INNER)
        Fint = ufl.dot(self.sig, self.eps4(v)) * self.w * self.dx
        self.F = fem.form(Fint - Fext if control == "pressure" else Fint)
        self.J = fem.form(ufl.dot(ufl.dot(self.Ct, self.eps4(du)), self.eps4(v)) * self.w * self.dx)
        # Dirichlet conditions: symmetry (2D); for displacement control also the inner surface
        fdim = self.dim - 1
        self.bcs = []
        if self.dim == 2:
            for tag, comp in ((BOTTOM, 1), (LEFT, 0)):
                dofs = fem.locate_dofs_topological(self.V.sub(comp), fdim, ft.find(tag))
                self.bcs.append(fem.dirichletbc(0.0, dofs, self.V.sub(comp)))
        self.g = fem.Function(self.V)                    # increment of the prescribed inner displacement
        if control == "displacement":
            dofs = fem.locate_dofs_topological(self.V, fdim, ft.find(INNER))
            self.bc_in = fem.dirichletbc(self.g, dofs)
            self.bcs.append(self.bc_in)
        # radial "lifting" field w_R = e_r (ro - r)/(ro - ri): the consistent reaction R(w_R) = p * ri * (pi/2 in 2D, 1 per radian in 1D)
        self.wR = fem.Function(self.V)
        if self.dim == 2:
            self.wR.interpolate(lambda X: np.vstack([X[0], X[1]]) / np.hypot(X[0], X[1]) * (ro - np.hypot(X[0], X[1])) / (ro - ri))
        else:
            self.wR.interpolate(lambda X: (ro - X[0]) / (ro - ri))
        self.R_form = fem.form(ufl.dot(self.sig, self.eps4(self.wR)) * self.w * self.dx)
        self.ur_form = fem.form((ufl.dot(self.u, -n) if self.dim == 2 else self.u) * self.ds(INNER))
        self.arc = (np.pi / 2 * ri) if self.dim == 2 else 1.0
        self.A = create_matrix(self.J); self.b = create_vector(self.V)
        self._new_ksp()
        self.load, self.ref = 0.0, 0.0

    def _new_ksp(self):
        self.ksp = PETSc.KSP().create(comm); self.ksp.setOperators(self.A)
        self.ksp.setType("preonly"); self.ksp.getPC().setType("lu"); self.ksp.getPC().setFactorSolverType("mumps")

    def eps4(self, u):
        """strain in Mandel form (xx, yy, zz, sqrt2 xy); 1D radial: (rr, theta theta, zz, 0). Plane strain: eps_zz = 0."""
        if self.dim == 2:
            e = ufl.sym(ufl.grad(u))
            return ufl.as_vector([e[0, 0], e[1, 1], 0, np.sqrt(2) * e[0, 1]])
        r = ufl.SpatialCoordinate(self.msh)[0]
        return ufl.as_vector([u.dx(0), u / r, 0, 0])

    def _constitutive(self):
        self.deps.interpolate(self.deps_expr)
        n4 = len(self.sig.x.array) // 4
        sig, p, Ct, pl = self.mat.update(self.deps.x.array.reshape(n4, 4), self.sig_n.x.array.reshape(n4, 4),
                                         self.p_n.x.array, self.tangent)
        self.sig.x.array[:] = sig.ravel(); self.p.x.array[:] = p; self.Ct.x.array[:] = Ct.ravel()
        return pl

    def _residual(self):
        """b = F(Du); constrained rows hold Du - (prescribed increment), so the first correction carries the boundary increment
        into the body with the converged tangent (lifting) instead of a jump in the boundary cells"""
        with self.b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(self.b, self.F)
        apply_lifting(self.b, [self.J], bcs=[self.bcs], x0=[self.Du.x.petsc_vec], alpha=-1.0)
        self.b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE)
        set_bc(self.b, self.bcs, self.Du.x.petsc_vec, -1.0)
        return self.b.norm()

    def step(self, load, rtol=1e-8, atol=1e-12, maxit=50, retry=True):
        """one load step to `load` (pressure, or inner radial displacement). Newton on the increment Du from the last
        converged state; converged when |R| <= rtol * (largest first residual so far) or |R| <= atol.
        If plain Newton fails and `retry`, the step is solved again from Du = 0 with backtracking (a correction that makes
        |R| grow is halved): near p_lim the tangent is almost singular, and round-off (threaded MUMPS) can throw a full
        step out of the convergence basin. `self.retried` says whether that happened. If both fail, the first attempt is reported.
        Returns (converged, iterations, residual history). Commits the state only if converged."""
        if self.control == "pressure":
            self.pload.value = load
        else:
            d = load - self.load                             # the increment of the inner radial displacement in this step
            if self.dim == 2:
                self.g.interpolate(lambda X: d * np.vstack([X[0], X[1]]) / np.hypot(X[0], X[1]))
            else:
                self.g.x.array[:] = d
        ok, it, hist = self._newton(rtol, atol, maxit, halvings=0)
        self.retried = False
        if not ok and retry:
            self.ksp.destroy(); self._new_ksp()              # a failed LU factorisation (singular tangent) stays in the KSP
            ok2, it2, hist2 = self._newton(rtol, atol, maxit, halvings=4)
            self.retried = True
            if ok2:
                ok, it, hist = ok2, it2, hist2
        if ok:
            self.u.x.array[:] += self.Du.x.array             # commit the converged step
            self.sig_n.x.array[:] = self.sig.x.array; self.p_n.x.array[:] = self.p.x.array
            self.load = load
        return ok, it, hist

    def _newton(self, rtol, atol, maxit, halvings):
        self.Du.x.array[:] = 0.0
        self._constitutive(); res = self._residual(); hist = [res]
        self.ref = max(self.ref, res)                        # scale: the largest first residual of any step so far
        it = 0
        while not (res <= rtol * self.ref or res <= atol):
            if it == maxit:
                return False, it, hist
            self.A.zeroEntries(); assemble_matrix(self.A, self.J, bcs=self.bcs); self.A.assemble()
            self.ksp.solve(self.b, self.du.x.petsc_vec); self.du.x.scatter_forward()
            if not comm.allreduce(bool(np.all(np.isfinite(self.du.x.array))), op=MPI.LAND):   # all ranks take the same branch
                return False, it, hist
            Du0, a = self.Du.x.array.copy(), 1.0
            for k in range(halvings + 1):                    # backtracking (halvings > 0): first step length that lowers |R|
                self.Du.x.array[:] = Du0 - a * self.du.x.array
                self._constitutive(); new = self._residual()
                if new < res or k == halvings:
                    break
                a *= 0.5
            res = new; hist.append(res); it += 1
        return True, it, hist

    def ur_inner(self):
        return comm.allreduce(fem.assemble_scalar(self.ur_form), op=MPI.SUM) / self.arc

    def pressure_from_reaction(self):
        """inner pressure from the consistent reaction (works for both controls; for pressure control it returns p)"""
        return comm.allreduce(fem.assemble_scalar(self.R_form), op=MPI.SUM) / (self.ri * self.arc)

    def plastic_radius(self):
        """largest radius of a quadrature point with p > 0 (ri if none is plastic)"""
        r = self.rq.x.array[self.p_n.x.array > 0.0]
        return comm.allreduce(r.max() if len(r) else self.ri, op=MPI.MAX)

    def max_von_mises(self):
        n4 = len(self.sig_n.x.array) // 4
        return comm.allreduce(von_mises(self.sig_n.x.array.reshape(n4, 4)).max(), op=MPI.MAX)

## 2 · One material point  *(lecture §A2–A4, §C1)*
No mesh, no DOLFINx: `update` acts on arrays. Three checks: (i) after the return every plastic point is **on** the yield surface; (ii) the consistent
tangent is the derivative of `update` (central differences); (iii) a strain-controlled uniaxial cycle.

In [ ]:
import sys; sys.path.insert(0, "/tmp/fnx")
import numpy as np, matplotlib.pyplot as plt
from j2 import *

E, nu, s0 = 70e3, 0.3, 250.0
m0, mH = J2(E, nu, s0), J2(E, nu, s0, H=E / 10)
rng = np.random.default_rng(4)
deps = rng.normal(size=(10000, 4)) * 4e-3                       # 10^4 random strain increments from sigma_n = 0
for name, m in (("H = 0", m0), ("H = E/10", mH)):
    s, p, Ct, pl = m.update(deps, np.zeros((10000, 4)), np.zeros(10000))
    f = von_mises(s) - (m.sig0 + m.H * p)
    print(f"{name:9s}: {pl.sum()} plastic points; after the return max|f|/sig0 = {np.abs(f[pl]).max() / s0:.1e};  elastic points: max f/sig0 = {f[~pl].max() / s0:.1e}")

d1, sn1, pn1 = np.array([[3e-3, -1e-3, 0, 2e-3]]), np.array([[100., -50, 20, 30 * np.sqrt(2)]]), np.array([0.01])
for tang in ("consistent", "elastic"):
    Ct = mH.update(d1, sn1, pn1, tang)[2][0]
    h, Cfd = 1e-8, np.zeros((4, 4))
    for j in range(4):
        e = d1.copy(); e[0, j] += h; sp = mH.update(e, sn1, pn1)[0]
        e = d1.copy(); e[0, j] -= h; sm = mH.update(e, sn1, pn1)[0]
        Cfd[:, j] = (sp - sm)[0] / (2 * h)
    print(f"{tang:10s} tangent vs central difference: max|C - C_fd| / max|C| = {np.abs(Ct - Cfd).max() / np.abs(mH.C).max():.1e}")

ey = s0 / E
path = np.r_[np.linspace(0, 4 * ey, 41)[1:], np.linspace(4 * ey, -4 * ey, 81)[1:], np.linspace(-4 * ey, 4 * ey, 81)[1:]]
fig, ax = plt.subplots(figsize=(6, 4))
for name, m in (("H = 0", m0), ("H = E/10", mH)):
    c = uniaxial(m, path)
    print(f"{name:9s}: sigma_xx at 4 eps_y = {c[39, 0]:.4f}   (sig0 + E_t * 3 eps_y = {s0 + E * m.H / (E + m.H) * 3 * ey:.4f});  local iterations <= {int(c[:, 2].max())}")
    ax.plot(path / ey, c[:, 0] / s0, label=name)
ax.set(xlabel="eps_xx / eps_y", ylabel="sigma_xx / sig0", title="one material point, uniaxial stress"); ax.grid(alpha=0.3); ax.legend(); plt.show()

> **✏️ Your turn (8 min).** (i) An earlier version of this course wrote the corrector as `sigma = sigma_tr - 2*mu*dgamma*n` with `n = s/sigma_eq` and the same
> `dgamma`. Predict $f$ after this "return" in terms of $f^{\rm tr}$, then check it: copy `update` into a cell, make that change, and repeat the first test.
> (ii) Why does the local Newton of `uniaxial` need only one iteration per step?

> *your answer*

In [ ]:
# TODO: (i) a copy of update() with n = s / sigma_eq and sigma = sigma_tr - 2*mu*dgamma*n; f after the return for the 10^4 random increments

## 3 · The tube: elastic check and first yield  *(lecture §A6, §C2)*
$r_i=1$ m, $r_o=1.3$ m, plane strain, $E=70$ GPa, $\nu=0.3$, $\sigma_0=250$ MPa, $H=0$, inner pressure $p$. The 2D model is a quarter with rollers on
the symmetry lines; the 1D model is the radial line with weight $r$.

> **✏️ Predict first (3 min).** The textbook formula $p_y=\frac{\sigma_0}{\sqrt3}\big(1-\frac{r_i^2}{r_o^2}\big)=58.93$ MPa. Is it the first-yield pressure of this tube? If not, higher or lower?

> *your prediction*

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plas import *

E, nu, s0 = 70e3, 0.3, 250.0                     # MPa (an aluminium alloy)
ri, ro = 1.0, 1.3                                # m
py, plim = p_first_yield(s0, ri, ro, nu), p_limit(s0, ri, ro)
uy = lame_ur(py, ri, ro, E, nu)                  # u_r(ri) at first yield

msh, ft = annulus_mesh(ri, ro, 8, 24)
T = Tube(msh, ft, J2(E, nu, s0), ri, ro)
ok, it, hist = T.step(0.5 * py)
say(f"elastic step p = 0.5 p_y: converged {ok} in {it} iteration(s);  u_r(ri) = {T.ur_inner():.8e}   Lame {lame_ur(0.5 * py, ri, ro, E, nu):.8e}")
say(f"p_y(nu = 0.3) = {py:.4f} MPa;  nu = 0.5 formula {p_first_yield(s0, ri, ro, 0.5):.4f} MPa;  p_lim = {plim:.4f} MPa")
for lab, pv in (("0.999 p_y", 0.999 * py), ("1.001 p_y", 1.001 * py), ("0.999 * 58.93", 0.999 * p_first_yield(s0, ri, ro, 0.5))):
    T1 = Tube(*radial_mesh(ri, ro, 400), J2(E, nu, s0), ri, ro)
    ok, it, hist = T1.step(pv)
    npl = comm.allreduce(int((T1.p_n.x.array > 0).sum()), op=MPI.SUM)
    say(f"1D, p = {lab:14s} = {pv:.4f}: converged {ok}, plastic quadrature points {npl}, plastic radius {T1.plastic_radius():.4f}")

> **✏️ Your turn (5 min).** From $\sigma_{rr},\sigma_{\theta\theta}$ (Lamé) and $\sigma_{zz}=2\nu A$ at $r=r_i$, show $\sigma_{\rm eq}^2=3B^2/r_i^4+(1-2\nu)^2A^2$. Which term does the
> textbook formula drop, and for which material is it exact? Why is the inner surface the first to yield?

> *your answer*

## 4 · Load–displacement and Newton: two tangents  *(lecture §A4, §B4, §C3–C4)*
20 equal pressure steps from 0 to $0.99\,p_{\rm lim}$, tolerance $10^{-8}$. Same mesh, same load, two tangents; the 1D model as reference.
The elastic-tangent run is allowed 150 iterations per step (it takes a while).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plas import *

E, nu, s0 = 70e3, 0.3, 250.0                     # MPa (an aluminium alloy)
ri, ro = 1.0, 1.3                                # m
py, plim = p_first_yield(s0, ri, ro, nu), p_limit(s0, ri, ro)
uy = lame_ur(py, ri, ro, E, nu)                  # u_r(ri) at first yield

loads = np.linspace(0, 0.99 * plim, 21)[1:]
msh, ft = annulus_mesh(ri, ro, 8, 24)
out = {}
for name, mk, tang, maxit in (("2D consistent", lambda: (msh, ft), "consistent", 50), ("2D elastic", lambda: (msh, ft), "elastic", 150),
                              ("1D consistent", lambda: radial_mesh(ri, ro, 400), "consistent", 50)):
    T = Tube(*mk(), J2(E, nu, s0), ri, ro, tangent=tang)
    its, cur = [], [(0.0, 0.0)]
    for pv in loads:
        ok, it, hist = T.step(pv, maxit=maxit)
        its.append(it if ok else f"FAILED({it})")
        if not ok:
            break
        cur.append((pv, T.ur_inner()))
    out[name.replace(" ", "_")] = np.array(cur)
    say(f"{name:14s}: iterations of steps 15-20 {its[14:]}   last converged p = {cur[-1][0]:.3f}, u_r(ri)/u_y = {cur[-1][1] / uy:.5f}")
    if name == "2D consistent":
        say("   residual / first residual in the last step:", " ".join(f"{x / hist[0]:.1e}" for x in hist))
if comm.rank == 0:
    np.savez("/tmp/fnx/w04_curve.npz", uy=uy, py=py, plim=plim, **out)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
D = np.load("/tmp/fnx/w04_curve.npz"); uy = float(D["uy"])
fig, ax = plt.subplots(figsize=(6, 4))
for k, sty in (("2D_consistent", "o-"), ("1D_consistent", "k-"), ("2D_elastic", "x")):
    ax.plot(D[k][:, 1] / uy, D[k][:, 0], sty, ms=4, lw=1, label=k.replace("_", " "))
ax.axhline(float(D["plim"]), color="r", ls="--", lw=0.8, label="p_lim"); ax.axhline(float(D["py"]), color="g", ls=":", lw=0.8, label="p_y")
ax.set(xlabel="u_r(ri) / u_y", ylabel="p [MPa]"); ax.grid(alpha=0.3); ax.legend(); plt.show()

> **✏️ Your turn (5 min).** (i) From the printed residuals of the last consistent step, is the convergence quadratic? (ii) Why does the elastic tangent need more
> iterations step after step, and why does it fail near $p_{\rm lim}$? (iii) When could an elastic (or "secant") tangent still be a reasonable choice?

> *your answer*

## 5 · The limit load: pressure control vs displacement control  *(lecture §A7, §B5, §C6)*
Pressure control: step towards and past $p_{\rm lim}$. Displacement control: prescribe $u_r(r_i)$ up to $50\,u_y$ and read $p$ from the consistent reaction.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plas import *

E, nu, s0 = 70e3, 0.3, 250.0                     # MPa (an aluminium alloy)
ri, ro = 1.0, 1.3                                # m
py, plim = p_first_yield(s0, ri, ro, nu), p_limit(s0, ri, ro)
uy = lame_ur(py, ri, ro, E, nu)                  # u_r(ri) at first yield

T = Tube(*radial_mesh(ri, ro, 400), J2(E, nu, s0), ri, ro)
for pv in (70.0, 75.0, 75.5, 75.7, 75.73, 1.001 * plim):
    ok, it, hist = T.step(pv)
    say(f"pressure control p = {pv:7.3f}: converged {ok}, iterations {it:2d}, u_r(ri)/u_y = {T.ur_inner() / uy:.3f}, last residual {hist[-1]:.1e}")
fs = (0.5, 1, 1.5, 2, 3, 5, 8, 12, 20, 30, 50)
for name, mk in (("1D 400", lambda: radial_mesh(ri, ro, 400)), ("2D 8x24", lambda: annulus_mesh(ri, ro, 8, 24))):
    T = Tube(*mk(), J2(E, nu, s0), ri, ro, control="displacement")
    row = []
    for f in fs:
        ok, it, hist = T.step(f * uy, rtol=1e-12)   # tight: the lifting inflates the reference residual
        row.append(f"{T.pressure_from_reaction():.6f}" + ("" if ok else "(FAILED)"))
    say(f"displacement control {name}: p at u_r/u_y = {fs}:\n   {row}")
say(f"exact p_lim = {plim:.6f} MPa")

> **✏️ Your turn (5 min).** (i) What does the solver report at $1.001\,p_{\rm lim}$, and what would a code that ignores the flag print as the "result"?
> (ii) Why can displacement control pass the limit load? (iii) The 2D values at $50\,u_y$ come out slightly **above** $p_{\rm lim}$. Why is that not a contradiction?

> *your answer*

## 6 · LLM exercise: the plastic zone against Hill's solution  *(lecture Part D)*
Ask your LLM for a cell that uses `plas.py` (paste it, `j2.py`, and the API sheet §16) to follow the elastic–plastic boundary $c(p)$ of the tube and compare it
with Hill's closed-form solution.

```text
Using the attached modules j2.py and plas.py (DOLFINx 0.11), write a %%fenicsx -np 2 cell for the thick tube ri = 1, ro = 1.3, plane strain,
E = 70e3, sig0 = 250, H = 0, under inner pressure. Use the 1D radial model (radial_mesh, 400 cells). For nu = 0.3 and nu = 0.45, load from the
first-yield pressure to 0.98 p_lim in 14 steps and print the plastic radius c at each step next to Hill's solution
p = sig0/sqrt(3) * (2 ln(c/ri) + 1 - c^2/ro^2). Say what Hill assumes and explain the difference.
```

```python
# paste here
```

**Check before you trust it**

- [ ] Every load step reports converged = True before $c$ is read; $c$ comes from the **converged** state
- [ ] The history (`sig_n`, `p_n`) is updated once per converged step, not after every Newton correction: twice the load steps must leave $u_r(r_i)$ unchanged (the per-correction update converges and keeps $f\le0$, so neither check catches it)
- [ ] The load range starts at $p_y(\nu)$ of each material (not at the $\nu=\tfrac12$ value)
- [ ] Hill's assumption is stated: what is $\sigma_{zz}$ in the elastic zone in his solution, and in yours?
- [ ] $c$ is the radius of a quadrature point: its resolution is about one cell; repeat with 800 cells
- [ ] `-np 1` and `-np 2` give the same numbers

In [ ]:
%%fenicsx -np 2
# TODO: the LLM's cell (after you checked it)

> *your verdict on the LLM's code*